In [1]:
# --- CELL 1: ENVIRONMENT & DEPENDENCIES ---
%%capture
!pip install facenet-pytorch librosa torchvision opencv-python-headless

import os
import cv2
import torch
import warnings
import numpy as np
import torch.nn as nn
import librosa
from typing import Tuple, Optional, Dict
from PIL import Image
from torchvision import models, transforms
from facenet_pytorch import MTCNN

# Suppress warnings for clean production output
warnings.filterwarnings('ignore')

# Global Device Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
# --- CELL 2: NEURAL NETWORK ARCHITECTURES ---
import torch.nn.functional as F

# -----------------------------------------
# A. Audio Model (Custom SE-ResNet)
# -----------------------------------------
class SEResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1),
                nn.BatchNorm2d(out_channels)
            )

        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(out_channels, max(1, out_channels // 8)),
            nn.ReLU(),
            nn.Linear(max(1, out_channels // 8), out_channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv(x)
        y = self.se(out).unsqueeze(-1).unsqueeze(-1)
        return F.relu(out * y + residual)

class AudioDeepfakeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)

        self.res1 = SEResidualBlock(64, 64)
        self.drop1 = nn.Dropout2d(0.2)

        self.res2 = SEResidualBlock(64, 128)
        self.drop2 = nn.Dropout2d(0.3)

        self.res3 = SEResidualBlock(128, 256)
        self.drop3 = nn.Dropout2d(0.4)

        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.7),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        x = F.relu(self.bn1(self.layer1(x)))
        x = self.res1(x)
        x = self.drop1(x)
        x = self.res2(x)
        x = self.drop2(x)
        x = self.res3(x)
        x = self.drop3(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

# -----------------------------------------
# B. Video Model (Vision Transformer)
# -----------------------------------------
class VideoDeepfakeModel(nn.Module):
    def __init__(self, num_classes=2):
        super(VideoDeepfakeModel, self).__init__()
        # We use weights=None because we are loading your custom weights later
        self.model = models.vit_b_16(weights=None)
        self.model.heads = nn.Sequential(
            nn.Linear(768, num_classes)
        )

    def forward(self, x):
        return self.model(x)

print("Custom SE-ResNet Audio & ViT Video Architectures Defined!")

Custom SE-ResNet Audio & ViT Video Architectures Defined!


In [9]:
# --- CELL 3: THE MULTIMODAL FUSION ENGINE CLASS ---

class DeepfakeFusionEngine:
    """
    Enterprise-grade engine that loads Visual and Acoustic models to evaluate media integrity.
    Utilizes an OR-gate security cascading logic for maximum threat detection.
    """
    def __init__(self, video_weights_path: str, audio_weights_path: str, device: torch.device):
        self.device = device
        self.threshold = 0.50

        # Initialize Transformers and Detectors
        self.val_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.mtcnn = MTCNN(margin=40, keep_all=False, post_process=False, device=self.device)

        # Load Sub-Systems
        self._load_video_subsystem(video_weights_path)
        self._load_audio_subsystem(audio_weights_path)
        print("🛡️ Fusion Engine Online: All Neural Sub-systems loaded successfully.")

    def _load_video_subsystem(self, path: str):
        self.video_model = VideoDeepfakeModel(num_classes=2).to(self.device)
        self.video_model.load_state_dict(torch.load(path, map_location=self.device, weights_only=True))
        self.video_model.eval()

    def _load_audio_subsystem(self, path: str):
        self.audio_model = AudioDeepfakeModel().to(self.device)
        self.audio_model.load_state_dict(torch.load(path, map_location=self.device, weights_only=True))
        self.audio_model.eval()

    def analyze_video(self, video_path: str, num_frames: int = 10) -> Optional[float]:
        """Extracts faces across time-steps and returns mean probability of being REAL."""
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0: return None

        interval = max(1, total_frames // num_frames)
        face_probs = []

        for i in range(num_frames):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i * interval)
            success, frame = cap.read()
            if not success: continue

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            face_tensor = self.mtcnn(Image.fromarray(frame_rgb))

            if face_tensor is not None:
                face_np = face_tensor.permute(1, 2, 0).byte().cpu().numpy()
                face_transformed = self.val_transform(Image.fromarray(face_np)).unsqueeze(0).to(self.device)

                with torch.no_grad():
                    logits = self.video_model(face_transformed)
                    prob_real = torch.softmax(logits, dim=1)[:, 1].item()
                    face_probs.append(prob_real)

        cap.release()
        return sum(face_probs) / len(face_probs) if face_probs else None

    def analyze_audio(self, video_path: str) -> Optional[float]:
        """Extracts audio track, computes mel-spectrogram, and returns probability of being REAL."""
        try:
            y, sr = librosa.load(video_path, sr=16000, duration=3.0)
            if len(y) == 0: return None
        except Exception:
            return None # Audio track missing or corrupt

        target_samples = 16000 * 3
        y = np.pad(y, (0, target_samples - len(y))) if len(y) < target_samples else y[:target_samples]

        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, hop_length=375, n_fft=1024)
        mel_db = librosa.power_to_db(mel, ref=np.max)[:, :128]

        tensor = torch.tensor(np.stack([mel_db, librosa.feature.delta(mel_db), librosa.feature.delta(mel_db, order=2)])).float()
        for c in range(tensor.size(0)):
            mean, std = tensor[c].mean(), tensor[c].std()
            tensor[c] = (tensor[c] - mean) / std if std > 1e-6 else tensor[c] - mean

        with torch.no_grad():
            logits = self.audio_model(tensor.unsqueeze(0).to(self.device))
            prob_real = torch.sigmoid(logits).item()

        return prob_real

    def scan_media(self, video_path: str) -> Dict[str, any]:
        """Executes the full pipeline and returns a structured diagnosis dictionary."""
        vid_prob = self.analyze_video(video_path)
        aud_prob = self.analyze_audio(video_path)

        # Failsafes if modalities are missing
        vid_prob = vid_prob if vid_prob is not None else 0.50
        aud_prob = aud_prob if aud_prob is not None else 0.50

        is_fake = (vid_prob < self.threshold) or (aud_prob < self.threshold)

        diagnosis = "Genuine Media"
        if is_fake:
            if vid_prob < self.threshold and aud_prob < self.threshold:
                diagnosis = "Full Synthesis (AI Face + AI Voice)"
            elif vid_prob < self.threshold:
                diagnosis = "Visual Manipulation (Face-swap / Lip-sync)"
            else:
                diagnosis = "Acoustic Manipulation (Voice Cloning)"

        return {
            "video_confidence": vid_prob,
            "audio_confidence": aud_prob,
            "is_deepfake": is_fake,
            "diagnosis": diagnosis
        }

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
# --- CELL 4: EXECUTION & REPORTING ---

def print_security_report(file_path: str, results: dict):
    """Formats the results dictionary into a professional CLI security report."""
    v_conf = results['video_confidence']
    a_conf = results['audio_confidence']

    v_percent = (v_conf if v_conf > 0.5 else 1 - v_conf) * 100
    a_percent = (a_conf if a_conf > 0.5 else 1 - a_conf) * 100

    v_label = "REAL" if v_conf > 0.5 else "FAKE"
    a_label = "REAL" if a_conf > 0.5 else "FAKE"

    print("\n" + "═"*65)
    print(f" MULTIMODAL INTEGRITY SCAN REPORT")
    print("═"*65)
    print(f" Target File  : {os.path.basename(file_path)}")
    print("─"*65)
    print(f" Vision Sub-System  : {v_label:<5} | Confidence: {v_percent:.1f}%")
    print(f" Audio Sub-System   : {a_label:<5} | Confidence: {a_percent:.1f}%")
    print("─"*65)

    if results['is_deepfake']:
        print(f" SYSTEM VERDICT    : DEEPFAKE DETECTED")
        print(f" DIAGNOSIS         : {results['diagnosis']}")
    else:
        print(f" SYSTEM VERDICT    : GENUINE MEDIA")
        print(f" DIAGNOSIS         : Passed all security protocols.")
    print("═"*65 + "\n")

# ==========================================
#  INITIALIZE AND RUN THE ENGINE
# ==========================================
# Update these paths to your specific Google Drive locations
VIDEO_WEIGHTS = '/content/drive/MyDrive/data/celeb_df/models/best_video_detector.pth'
AUDIO_WEIGHTS = '/content/drive/MyDrive/deepfake_audio_research/checkpoints/best_audio_detector_v2.pth'
TEST_VIDEO = '/content/drive/MyDrive/data/celeb_df/test_video.mp4'

# 1. Boot up the engine (Only needs to happen once!)
engine = DeepfakeFusionEngine(VIDEO_WEIGHTS, AUDIO_WEIGHTS, DEVICE)

# 2. Run the scan and print the report
results = engine.scan_media(TEST_VIDEO)
print_security_report(TEST_VIDEO, results)

🛡️ Fusion Engine Online: All Neural Sub-systems loaded successfully.

═════════════════════════════════════════════════════════════════
 MULTIMODAL INTEGRITY SCAN REPORT
═════════════════════════════════════════════════════════════════
 Target File  : test_video.mp4
─────────────────────────────────────────────────────────────────
 Vision Sub-System  : REAL  | Confidence: 88.5%
 Audio Sub-System   : REAL  | Confidence: 100.0%
─────────────────────────────────────────────────────────────────
 SYSTEM VERDICT    : GENUINE MEDIA
 DIAGNOSIS         : Passed all security protocols.
═════════════════════════════════════════════════════════════════

